In [25]:
# pip install pandas numpy matplotlib seaborn scikit-learn

# Bước 1: Khởi tạo và Load dữ liệu

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

def load_loan_data(file_path):
    """Load dữ liệu từ file CSV."""
    try:
        df = pd.read_csv(file_path, index_col=0).reset_index()
        print(f"✅ Đã load dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột.")
        return df
    except Exception as e:
        print(f"❌ Lỗi khi load file: {e}")
        return None


# Bước 2: Data Understanding

In [27]:
def get_data_health_report(df):
    """Kiểm tra Missing values, Duplicates và Kiểu dữ liệu."""
    info_df = pd.DataFrame({
        'Data Type': df.dtypes,
        'Missing Values': df.isnull().sum(),
        'Missing %': (df.isnull().sum() / len(df) * 100).round(2),
        'Unique Values': df.nunique()
    })
    
    print("--- BÁO CÁO SỨC KHỎE DỮ LIỆU ---")
    print(f"Số dòng trùng lặp: {df.duplicated().sum()}")
    return info_df

def check_target_balance(df, target_col='Default'):
    """Kiểm tra tỷ lệ nợ xấu."""
    balance = df[target_col].value_counts(normalize=True) * 100
    print(f"\n--- TỶ LỆ BIẾN MỤC TIÊU ({target_col}) ---")
    print(f"Trả nợ tốt (0): {balance[0]:.2f}%")
    print(f"Vỡ nợ (1): {balance[1]:.2f}%")

# Bước 3: Data Preparation

## 3.1 Xử lý giá trị thiếu và Outliers

In [28]:
def handle_missing_values(df):
    """Điền giá trị thiếu: Median cho số, Mode cho phân loại."""
    df_clean = df.copy()
    num_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns
    cat_cols = df_clean.select_dtypes(include=['object']).columns
    
    for col in num_cols:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        
    for col in cat_cols:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
        
    print("✅ Đã xử lý Missing values.")
    return df_clean

def cap_outliers(df, columns):
    """Giới hạn Outliers bằng phương pháp IQR (Capping)."""
    df_cap = df.copy()
    for col in columns:
        Q1 = df_cap[col].quantile(0.25)
        Q3 = df_cap[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df_cap[col] = np.where(df_cap[col] > upper_bound, upper_bound, 
                               np.where(df_cap[col] < lower_bound, lower_bound, df_cap[col]))
    print(f"✅ Đã xử lý Outliers cho các cột: {columns}")
    return df_cap

## 3.2 Mã hóa biến Categorical (Encoding)

In [29]:
def encode_categorical_features(df):
    """Mã hóa các biến định tính sang định lượng."""
    df_encoded = df.copy()
    
    # 1. Label Encoding cho Education (có thứ tự)
    edu_map = {'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3}
    if 'Education' in df_encoded.columns:
        df_encoded['Education'] = df_encoded['Education'].map(edu_map)
    
    # 2. One-Hot Encoding cho các biến còn lại
    categorical_cols = ['EmploymentType', 'MaritalStatus', 'LoanPurpose']
    df_encoded = pd.get_dummies(df_encoded, columns=categorical_cols, drop_first=True)
    
    # 3. Chuyển Binary (HasMortgage, HasCoSigner...) sang 0/1
    binary_cols = [col for col in df_encoded.columns if col.startswith('Has')]
    for col in binary_cols:
        if df_encoded[col].dtype == 'object':
            df_encoded[col] = df_encoded[col].map({'Yes': 1, 'No': 0, 'Y': 1, 'N': 0})
            
    print("✅ Đã mã hóa các biến Categorical.")
    return df_encoded

# Bước 4: Chia dữ liệu

In [30]:
def split_and_save_data(df, target='Default', test_size=0.2):
    """Chia tập Train/Test và lưu trữ."""
    X = df.drop(columns=['LoanID', target]) # Loại bỏ ID và Target
    y = df[target]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )
    
    print(f"✅ Chia dữ liệu thành công. Kích thước tập Train: {X_train.shape}")
    return X_train, X_test, y_train, y_test

## Bước 5: Lưu dữ liệu

In [31]:
import os
import joblib

def save_processed_data(X_train, X_test, y_train, y_test, folder_path='../data/processed'):
    """Lưu tập Train/Test vào thư mục processed."""
    
    # Tạo thư mục nếu chưa tồn tại
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"📁 Đã tạo thư mục mới: {folder_path}")

    # Cách 1: Lưu dạng CSV (Dễ đọc, dễ kiểm tra bằng Excel)
    X_train.to_csv(f'{folder_path}/X_train.csv', index=False)
    X_test.to_csv(f'{folder_path}/X_test.csv', index=False)
    y_train.to_csv(f'{folder_path}/y_train.csv', index=False)
    y_test.to_csv(f'{folder_path}/y_test.csv', index=False)
    
    # Cách 2: Lưu dạng Joblib (Nhanh, giữ nguyên định dạng dữ liệu chuẩn của Python)
    data_bundle = {
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test
    }
    joblib.dump(data_bundle, f'{folder_path}/processed_data_bundle.joblib')
    
    print(f"💾 Đã lưu 4 file (X_train, X_test, y_train, y_test) vào: {folder_path}")

In [ ]:
import joblib
import os

def load_data_bundle(file_path='./data/processed/processed_data_bundle.joblib'):
    """
    Load toàn bộ tập Train/Test từ file joblib và in ra kích thước kiểm tra.
    """
    if not os.path.exists(file_path):
        print(f"❌ Không tìm thấy file tại: {file_path}")
        return None
    
    # Load bundle
    data = joblib.load(file_path)
    
    X_train = data['X_train']
    X_test = data['X_test']
    y_train = data['y_train']
    y_test = data['y_test']
    
    # Kiểm tra kích thước
    print("--- KIỂM TRA KÍCH THƯỚC DỮ LIỆU ---")
    print(f"Shape X_train: {X_train.shape} | Shape y_train: {y_train.shape}")
    print(f"Shape X_test:  {X_test.shape}  | Shape y_test:  {y_test.shape}")
    
    # Kiểm tra tỷ lệ nợ xấu để đảm bảo Stratified Split hoạt động đúng
    train_default_rate = (y_train.sum() / len(y_train) * 100).round(2)
    test_default_rate = (y_test.sum() / len(y_test) * 100).round(2)
    
    print(f"\n--- TỶ LỆ NỢ XẤU (DEFAULT RATE) ---")
    print(f"Trong tập Train: {train_default_rate}%")
    print(f"Trong tập Test:  {test_default_rate}%")
    
    return X_train, X_test, y_train, y_test

# --- THỰC THI KIỂM TRA ---
# X_train, X_test, y_train, y_test = load_data_bundle()

# Main Flow

In [32]:
# --- CÀI ĐẶT ĐƯỜNG DẪN ---
INPUT_PATH = './data/raw/raw_data.csv'
OUTPUT_FOLDER = './data/processed'

# --- THỰC THI STEP-BY-STEP ---

# Bước 1: Load dữ liệu
raw_df = load_loan_data(INPUT_PATH)

if raw_df is not None:
    # Bước 2: Understanding (Báo cáo sức khỏe)
    display(get_data_health_report(raw_df))
    check_target_balance(raw_df)

    # Bước 3: Preparation (Tiền xử lý)
    # 3.1 Xử lý giá trị khuyết
    df_no_missing = handle_missing_values(raw_df)
    
    # 3.2 Xử lý giá trị ngoại lai cho các cột số quan trọng
    cols_to_cap = ['Income', 'LoanAmount', 'CreditScore', 'InterestRate']
    df_capped = cap_outliers(df_no_missing, cols_to_cap)
    
    # 3.3 Mã hóa biến phân loại
    df_final = encode_categorical_features(df_capped)

    # Bước 4: Chia dữ liệu (Split)
    X_train, X_test, y_train, y_test = split_and_save_data(df_final)

    # Bước 5: Lưu dữ liệu (Save)
    save_processed_data(X_train, X_test, y_train, y_test, OUTPUT_FOLDER)

    print("\n🚀 QUY TRÌNH DATA PREPARATION HOÀN TẤT!")

✅ Đã load dữ liệu: 255347 dòng, 18 cột.
--- BÁO CÁO SỨC KHỎE DỮ LIỆU ---
Số dòng trùng lặp: 0


,Data Type,Missing Values,Missing %,Unique Values
LoanID,str,0,0.0,255347
Age,int64,0,0.0,52
Income,int64,0,0.0,114620
LoanAmount,int64,0,0.0,158729
CreditScore,int64,0,0.0,550
MonthsEmployed,int64,0,0.0,120
NumCreditLines,int64,0,0.0,4
InterestRate,float64,0,0.0,2301
LoanTerm,int64,0,0.0,5
DTIRatio,float64,0,0.0,81



--- TỶ LỆ BIẾN MỤC TIÊU (Default) ---
Trả nợ tốt (0): 88.39%
Vỡ nợ (1): 11.61%


C:\Users\DELL\AppData\Local\Temp\ipykernel_15804\2309778015.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_clean.select_dtypes(include=['object']).columns


✅ Đã xử lý Missing values.
✅ Đã xử lý Outliers cho các cột: ['Income', 'LoanAmount', 'CreditScore', 'InterestRate']
✅ Đã mã hóa các biến Categorical.
✅ Chia dữ liệu thành công. Kích thước tập Train: (204277, 22)
📁 Đã tạo thư mục mới: ./data/processed
💾 Đã lưu 4 file (X_train, X_test, y_train, y_test) vào: ./data/processed

🚀 QUY TRÌNH DATA PREPARATION HOÀN TẤT!


In [ ]:
# --- THỰC THI KIỂM TRA ---
X_train, X_test, y_train, y_test = load_data_bundle()

--- KIỂM TRA KÍCH THƯỚC DỮ LIỆU ---
Shape X_train: (204277, 22) | Shape y_train: (204277,)
Shape X_test:  (51070, 22)  | Shape y_test:  (51070,)

--- TỶ LỆ NỢ XẤU (DEFAULT RATE) ---
Trong tập Train: 11.61%
Trong tập Test:  11.61%
